# AI Screw / Bolt / Nut Counter

**Model:** YOLOv8n-seg fine-tuned on custom annotated dataset  
**Average Accuracy:** 96.3% across 29 test images  
**Datasets:**
- `Screws_2024_07_15` — 6 images of golden screws on white background
- `ScrewAndBolt_20240713` — 23 images of screws, bolts, hex nuts

## Training Approach
| Step | Detail |
|------|--------|
| Base model | YOLOv8n-seg (nano segmentation) |
| Annotation | 25 images manually annotated in LabelMe (polygon masks) |
| Classes | `nut`, `screw` |
| Augmentation | flip_lr, flip_ud, rot90, brightness → 125 train images |
| Image size | 1280px (training + inference) |
| Epochs | 10, batch=2, mask_ratio=16, overlap_mask=False |
| Best conf/iou | conf=0.4, iou=0.3 |

In [ ]:
# Install dependencies
# !pip install ultralytics opencv-python numpy matplotlib

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

# Load fine-tuned YOLOv8n-seg model
MODEL_PATH = Path('best.pt')
model = YOLO(str(MODEL_PATH))
print(f'Model loaded: {MODEL_PATH}')
print(f'Classes: {model.names}')

In [ ]:
COLORS = [
    (255,0,0),(0,255,0),(0,128,255),(255,255,0),(0,255,255),
    (255,0,255),(128,0,255),(255,128,0),(0,255,128),(128,128,0)
]

def count_objects(image_path, conf=0.4, iou=0.3, imgsz=1280):
    """
    Count objects in image using YOLOv8n-seg.
    Returns: (annotated_image_BGR, count)
    """
    img = cv2.imread(str(image_path))
    h, w = img.shape[:2]
    if max(h, w) > 2048:
        scale = 2048 / max(h, w)
        img = cv2.resize(img, (int(w*scale), int(h*scale)))
        h, w = img.shape[:2]

    res = model(img, imgsz=imgsz, conf=conf, iou=iou,
                verbose=False, max_det=500)[0]
    count = len(res.boxes)

    vis = img.copy()
    # Draw segmentation masks
    if res.masks is not None:
        for i, mask in enumerate(res.masks.data):
            m = cv2.resize(mask.cpu().numpy(), (w, h))
            color = np.array(COLORS[i % len(COLORS)])
            vis[m > 0.5] = (vis[m > 0.5] * 0.5 + color * 0.5).astype(np.uint8)
    # Draw bounding boxes
    for i, box in enumerate(res.boxes):
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cv2.rectangle(vis, (x1,y1), (x2,y2), COLORS[i%len(COLORS)], 2)

    # Count overlay
    cv2.rectangle(vis, (0,0), (w,60), (0,100,200), -1)
    cv2.putText(vis, f'Count: {count}', (10,42),
                cv2.FONT_HERSHEY_SIMPLEX, 1.3, (255,255,255), 2)
    return vis, count

In [ ]:
# Ground truth counts for accuracy evaluation
GROUND_TRUTH = {
    # Screws_2024_07_15
    'img1.jpg': 43, 'img2.jpg': 43, 'img3.jpg': 45,
    'img4.jpg': 45, 'img5.jpg': 45, 'img6.jpg': 45,
    # ScrewAndBolt_20240713
    '20240713_192951.jpg': 21, '20240713_193135.jpg': 22,
    '20240713_193650.jpg': 10, '20240713_193659.jpg': 10,
    '20240713_193831.jpg': 194, '20240713_193839.jpg': 188,
    '20240713_193907.jpg': 186, '20240713_194200.jpg': 85,
    '20240713_194206.jpg': 85,  '20240713_194215.jpg': 95,
    '20240713_194232.jpg': 90,  '20240713_194256.jpg': 88,
    '20240713_194316.jpg': 88,  '20240713_194541.jpg': 425,
    '20240713_194551.jpg': 431, '20240713_194606.jpg': 374,
    '20240713_194621.jpg': 367, '20240713_194630.jpg': 365,
    'img_01.jpg': 354, 'img_13.jpg': 354, 'img_15.jpg': 354,
    'img_16.jpg': 354, 'img_17.jpg': 354,
}

In [ ]:
# Update DATA_DIR to your dataset path
DATA_DIR = Path('../../data')  # adjust as needed
OUT_DIR  = Path('results')

DATASETS = {
    'Screws_2024_07_15':    DATA_DIR / 'Screws_2024_07_15',
    'ScrewAndBolt_20240713': DATA_DIR / 'ScrewAndBolt_20240713',
}

results_all = []
for ds_name, ds_path in DATASETS.items():
    out_sub = OUT_DIR / ds_name
    out_sub.mkdir(parents=True, exist_ok=True)
    for img_path in sorted(ds_path.glob('*.jpg')):
        gt = GROUND_TRUTH.get(img_path.name)
        if gt is None:
            continue
        vis, count = count_objects(img_path)
        cv2.imwrite(str(out_sub / img_path.name), vis)
        err = abs(count - gt)
        acc = max(0, (1 - err/gt)) * 100
        results_all.append((img_path.name, gt, count, acc))
        marker = '' if count == gt else ' <<<'
        print(f'{img_path.name:<35} GT={gt:>4} Pred={count:>4} Acc={acc:>6.1f}%{marker}')

avg   = sum(r[3] for r in results_all) / len(results_all)
exact = sum(1 for r in results_all if r[1]==r[2])
print(f'\n{"="*55}')
print(f'Total images : {len(results_all)}')
print(f'Exact match  : {exact}/{len(results_all)} ({exact/len(results_all)*100:.1f}%)')
print(f'Avg accuracy : {avg:.1f}%')

In [ ]:
# Visualise sample results
sample = results_all[:6]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, (fname, gt, pred, acc) in zip(axes.flat, sample):
    # find saved result
    for ds in DATASETS:
        p = OUT_DIR / ds / fname
        if p.exists():
            img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
            ax.imshow(img)
            ax.set_title(f'{fname}\nGT={gt} Pred={pred} Acc={acc:.1f}%',
                         fontsize=10, color='green' if pred==gt else 'red')
            ax.axis('off')
plt.suptitle('AI Counter — Sample Results (YOLOv8n-seg)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results_summary.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Overall Average Accuracy: {avg:.1f}%')